In [ ]:
from pathlib import Path
import shutil
import random
import pandas as pd

TRAIN_DIR = Path("/Users/mac/MeterReadAI/data/DataForTrain/recognition_merge_2/train")
VAL_DIR = Path("/Users/mac/MeterReadAI/data/DataForTrain/recognition_merge_2/val")

VAL_DIR.mkdir(parents=True, exist_ok=True)
(VAL_DIR / "images").mkdir(exist_ok=True)

files = list((TRAIN_DIR / "images").glob("*.*"))
random.seed(42)
random.shuffle(files)

val_files = files[int(len(files) * 0.9):]
val_names = {f.stem for f in val_files}

# Xử lý CSV
csv_file = TRAIN_DIR / "train_labels.csv"
if csv_file.exists():
    df = pd.read_csv(csv_file, dtype={"label": str})
    print(df.head())
    
    # Lọc theo tên file và tạo bản sao
    val_df = df[df['filename'].str.replace('.jpg', '').str.replace('.png', '').isin(val_names)].copy()
    train_df = df[~df['filename'].str.replace('.jpg', '').str.replace('.png', '').isin(val_names)].copy()
    
    val_df['label'] = val_df['label'].astype(str)
    train_df['label'] = train_df['label'].astype(str)
    
    # Lưu CSV mới
    val_df.to_csv(VAL_DIR / "val_labels.csv", index=False)
    train_df.to_csv(csv_file, index=False)
    
    print(f"Đã tạo val_labels.csv với {len(val_df)} dòng")
    print(f"Đã cập nhật train_labels.csv với {len(train_df)} dòng")

for img in val_files:
    shutil.move(img, VAL_DIR / "images" / img.name)
    label = TRAIN_DIR / "labels" / f"{img.stem}.txt"
    if label.exists():
        shutil.move(label, VAL_DIR / "labels" / label.name)

print(f"Đã chuyển {len(val_files)} ảnh sang val")
print(f"Train: {len(list((TRAIN_DIR / 'images').glob('*.*')))}")
print(f"Val: {len(list((VAL_DIR / 'images').glob('*.*')))}")
